Name: James Fisher
Date: March 29, 2026
Course: DDS-8555 (Predictive Analytics)
Assignment: Week 5, Kaggle Obesity Assignment

In [7]:
# Load libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, classification_report

# Load data
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)

display(train_df.head())
display(test_df.head())
display(sample_submission.head())


Train shape: (20758, 18)
Test shape: (13840, 17)
Sample submission shape: (13840, 2)


,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS
0,20758,Male,26.899886,1.848294,120.644178,yes,yes,2.938616,3.000000,Sometimes,no,2.825629,no,0.855400,0.000000,Sometimes,Public_Transportation
1,20759,Female,21.000000,1.600000,66.000000,yes,yes,2.000000,1.000000,Sometimes,no,3.000000,no,1.000000,0.000000,Sometimes,Public_Transportation
2,20760,Female,26.000000,1.643355,111.600553,yes,yes,3.000000,3.000000,Sometimes,no,2.621877,no,0.000000,0.250502,Sometimes,Public_Transportation
3,20761,Male,20.979254,1.553127,103.669116,yes,yes,2.000000,2.977909,Sometimes,no,2.786417,no,0.094851,0.000000,Sometimes,Public_Transportation
4,20762,Female,26.000000,1.627396,104.835346,yes,yes,3.000000,3.000000,Sometimes,no,2.653531,no,0.000000,0.741069,Sometimes,Public_Transportation


,id,NObeyesdad
0,20758,Normal_Weight
1,20759,Normal_Weight
2,20760,Normal_Weight
3,20761,Normal_Weight
4,20762,Normal_Weight


In [8]:
# Set static parameters for models
target_col = "NObeyesdad"
id_col = "id"

X = train_df.drop(columns=[target_col])
y = train_df[target_col]

X_test = test_df.copy()

# Separate features by type
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

# Usually we do not want to model on the ID field
if id_col in categorical_cols:
    categorical_cols.remove(id_col)
if id_col in numerical_cols:
    numerical_cols.remove(id_col)

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Categorical columns: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']
Numerical columns: ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']


In [9]:
# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes:")
for i, cls in enumerate(label_encoder.classes_):
    print(i, "->", cls)

Classes:
0 -> Insufficient_Weight
1 -> Normal_Weight
2 -> Obesity_Type_I
3 -> Obesity_Type_II
4 -> Obesity_Type_III
5 -> Overweight_Level_I
6 -> Overweight_Level_II


In [10]:
# Complete train/validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

X_train: (16606, 17)
X_valid: (4152, 17)


In [11]:
# Create reusable preprocessing data pipeline (imputation + encoding + scaling)

numeric_transformer_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# For models that do not require scaling
numeric_transformer_unscaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_scaled = ColumnTransformer(transformers=[
    ("num", numeric_transformer_scaled, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

preprocessor_unscaled = ColumnTransformer(transformers=[
    ("num", numeric_transformer_unscaled, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

In [ ]:
## Multinomial Logistic Regression
logit_model = Pipeline(steps=[
    ("preprocessor", preprocessor_scaled),
    ("classifier", LogisticRegression(
        solver="lbfgs",
        max_iter=5000,
        random_state=42
    ))
])

logit_model.fit(X_train.drop(columns=[id_col]), y_train)

logit_valid_pred = logit_model.predict(X_valid.drop(columns=[id_col]))

print("Multinomial Logistic Regression Accuracy:",
      accuracy_score(y_valid, logit_valid_pred))
print()
print(classification_report(
    y_valid,
    logit_valid_pred,
    target_names=label_encoder.classes_
))

Multinomial Logistic Regression Accuracy: 0.8684971098265896

                     precision    recall  f1-score   support

Insufficient_Weight       0.89      0.95      0.92       505
      Normal_Weight       0.87      0.82      0.84       617
     Obesity_Type_I       0.81      0.85      0.83       582
    Obesity_Type_II       0.93      0.96      0.95       650
   Obesity_Type_III       1.00      1.00      1.00       809
 Overweight_Level_I       0.75      0.71      0.73       485
Overweight_Level_II       0.73      0.71      0.72       504

           accuracy                           0.87      4152
          macro avg       0.85      0.85      0.85      4152
       weighted avg       0.87      0.87      0.87      4152



In [14]:
## Linear Discriminant Analysis
# Transform first (for dense arrays) and then fit separately.
X_train_lda = preprocessor_scaled.fit_transform(X_train.drop(columns=[id_col]))
X_valid_lda = preprocessor_scaled.transform(X_valid.drop(columns=[id_col]))
X_test_lda = preprocessor_scaled.transform(X_test.drop(columns=[id_col]))

# Convert sparse matrices to dense arrays if needed
if hasattr(X_train_lda, "toarray"):
    X_train_lda = X_train_lda.toarray()
    X_valid_lda = X_valid_lda.toarray()
    X_test_lda = X_test_lda.toarray()

lda_model = LinearDiscriminantAnalysis()
lda_model.fit(X_train_lda, y_train)

lda_valid_pred = lda_model.predict(X_valid_lda)

print("LDA Accuracy:", accuracy_score(y_valid, lda_valid_pred))
print()
print(classification_report(
    y_valid,
    lda_valid_pred,
    target_names=label_encoder.classes_
))

LDA Accuracy: 0.8229768786127167

                     precision    recall  f1-score   support

Insufficient_Weight       0.81      0.93      0.87       505
      Normal_Weight       0.79      0.72      0.75       617
     Obesity_Type_I       0.78      0.78      0.78       582
    Obesity_Type_II       0.91      0.94      0.93       650
   Obesity_Type_III       0.99      1.00      0.99       809
 Overweight_Level_I       0.68      0.60      0.64       485
Overweight_Level_II       0.66      0.68      0.67       504

           accuracy                           0.82      4152
          macro avg       0.80      0.81      0.80      4152
       weighted avg       0.82      0.82      0.82      4152



In [15]:
## Naive Bayes (GaussianNB)
# Transform first (for dense arrays) and then fit separately.
X_train_nb = preprocessor_unscaled.fit_transform(X_train.drop(columns=[id_col]))
X_valid_nb = preprocessor_unscaled.transform(X_valid.drop(columns=[id_col]))
X_test_nb = preprocessor_unscaled.transform(X_test.drop(columns=[id_col]))

if hasattr(X_train_nb, "toarray"):
    X_train_nb = X_train_nb.toarray()
    X_valid_nb = X_valid_nb.toarray()
    X_test_nb = X_test_nb.toarray()

nb_model = GaussianNB()
nb_model.fit(X_train_nb, y_train)

nb_valid_pred = nb_model.predict(X_valid_nb)

print("Naive Bayes Accuracy:", accuracy_score(y_valid, nb_valid_pred))
print()
print(classification_report(
    y_valid,
    nb_valid_pred,
    target_names=label_encoder.classes_
))

Naive Bayes Accuracy: 0.5869460500963392

                     precision    recall  f1-score   support

Insufficient_Weight       0.56      0.75      0.64       505
      Normal_Weight       0.49      0.20      0.28       617
     Obesity_Type_I       0.38      0.42      0.40       582
    Obesity_Type_II       0.50      0.96      0.66       650
   Obesity_Type_III       0.96      1.00      0.98       809
 Overweight_Level_I       0.60      0.24      0.34       485
Overweight_Level_II       0.51      0.30      0.38       504

           accuracy                           0.59      4152
          macro avg       0.57      0.55      0.52      4152
       weighted avg       0.59      0.59      0.55      4152



In [16]:
## Support Vector Machine (SVM)
svm_model = Pipeline(steps=[
    ("preprocessor", preprocessor_scaled),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        decision_function_shape="ovr",
        random_state=42
    ))
])

svm_model.fit(X_train.drop(columns=[id_col]), y_train)

svm_valid_pred = svm_model.predict(X_valid.drop(columns=[id_col]))

print("SVM Accuracy:", accuracy_score(y_valid, svm_valid_pred))
print()
print(classification_report(
    y_valid,
    svm_valid_pred,
    target_names=label_encoder.classes_
))

SVM Accuracy: 0.8810211946050096

                     precision    recall  f1-score   support

Insufficient_Weight       0.90      0.93      0.92       505
      Normal_Weight       0.86      0.83      0.84       617
     Obesity_Type_I       0.85      0.88      0.87       582
    Obesity_Type_II       0.96      0.97      0.96       650
   Obesity_Type_III       1.00      1.00      1.00       809
 Overweight_Level_I       0.74      0.70      0.72       485
Overweight_Level_II       0.77      0.77      0.77       504

           accuracy                           0.88      4152
          macro avg       0.87      0.87      0.87      4152
       weighted avg       0.88      0.88      0.88      4152



In [18]:
# Refit best model on full training data
X_full = train_df.drop(columns=[target_col])
y_full = label_encoder.transform(train_df[target_col])

# Logistic Regression on full data
logit_model.fit(X_full.drop(columns=[id_col]), y_full)

# LDA on full data
X_full_lda = preprocessor_scaled.fit_transform(X_full.drop(columns=[id_col]))
X_test_lda_full = preprocessor_scaled.transform(X_test.drop(columns=[id_col]))

if hasattr(X_full_lda, "toarray"):
    X_full_lda = X_full_lda.toarray()
    X_test_lda_full = X_test_lda_full.toarray()

lda_model.fit(X_full_lda, y_full)

# Naive Bayes on full data
X_full_nb = preprocessor_unscaled.fit_transform(X_full.drop(columns=[id_col]))
X_test_nb_full = preprocessor_unscaled.transform(X_test.drop(columns=[id_col]))

if hasattr(X_full_nb, "toarray"):
    X_full_nb = X_full_nb.toarray()
    X_test_nb_full = X_test_nb_full.toarray()

nb_model.fit(X_full_nb, y_full)

# SVM on full data
svm_model.fit(X_full.drop(columns=[id_col]), y_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [19]:
# Generate predictions for test set
logit_test_pred = logit_model.predict(X_test.drop(columns=[id_col]))
lda_test_pred = lda_model.predict(X_test_lda_full)
nb_test_pred = nb_model.predict(X_test_nb_full)
svm_test_pred = svm_model.predict(X_test.drop(columns=[id_col]))

logit_labels = label_encoder.inverse_transform(logit_test_pred)
lda_labels = label_encoder.inverse_transform(lda_test_pred)
nb_labels = label_encoder.inverse_transform(nb_test_pred)
svm_labels = label_encoder.inverse_transform(svm_test_pred)

# Create submission files
submission_logit = pd.DataFrame({
    "id": test_df[id_col],
    "NObeyesdad": logit_labels
})

submission_lda = pd.DataFrame({
    "id": test_df[id_col],
    "NObeyesdad": lda_labels
})

submission_nb = pd.DataFrame({
    "id": test_df[id_col],
    "NObeyesdad": nb_labels
})

submission_svm = pd.DataFrame({
    "id": test_df[id_col],
    "NObeyesdad": svm_labels
})

submission_logit.to_csv("submission_logistic_regression.csv", index=False)
submission_lda.to_csv("submission_lda.csv", index=False)
submission_nb.to_csv("submission_naive_bayes.csv", index=False)
submission_svm.to_csv("submission_svm.csv", index=False)